In [7]:
# train_global_lstm_prototype.py

import os
import pickle

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ─── CONFIG ──────────────────────────────────────────────────────────────
DATA_PATH = '../data/traffic_cleaned.csv'
SEQ_LEN     = 24
TEST_SPLIT  = 0.2
BATCH_SIZE  = 32
EPOCHS      = 5
PATIENCE    = 5
MODELS_DIR  = '../models' 
MODEL_FN    = 'global_lstm_4sites.h5'
SCALER_FN   = 'global_scaler_4sites.pkl'
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(MODELS_DIR, exist_ok=True)
np.random.seed(42)

# 1) Load & sort
df = pd.read_csv(DATA_PATH, parse_dates=['DateTime'])
df.sort_values(['SCATS Number','DateTime'], inplace=True)

# 2) Pick only 4 sites for prototype
site_ids = df['SCATS Number'].unique()
print(f"Using all {len(site_ids)} SCATS sites:", site_ids)

# 3) Build sliding windows for those 4 sites
Xw, yw = [], []
for site in site_ids:
    sub = df[df['SCATS Number']==site].set_index('DateTime')
    flows = sub['Traffic_flow'].values
    for i in range(len(flows) - SEQ_LEN):
        Xw.append(flows[i:i+SEQ_LEN])
        yw.append(flows[i+SEQ_LEN])

X = np.array(Xw)[:,:,None]   # (samples, SEQ_LEN,1)
y = np.array(yw)[:,None]     # (samples,1)
print(f"Built {len(X)} windows from 4 sites.")

# 4) Global scaling
scaler = MinMaxScaler()
flatX = X.reshape(-1,1)
scaler.fit(np.vstack([flatX, y]))
Xs = scaler.transform(flatX).reshape(X.shape)
ys = scaler.transform(y)

# 5) Train/test split
n = len(Xs)
sp = int(n * (1 - TEST_SPLIT))
X_train, X_test = Xs[:sp], Xs[sp:]
y_train, y_test = ys[:sp], ys[sp:]
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

# 6) Build LSTM
model = Sequential([
    LSTM(64, input_shape=(SEQ_LEN,1), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1, activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# 7) Train
es = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es],
    shuffle=False,
    verbose=2
)

# 8) Evaluate
mse, mae_s = model.evaluate(X_test, y_test, verbose=0)
y_pred_s = model.predict(X_test, verbose=0)
y_pred   = scaler.inverse_transform(y_pred_s)
y_true   = scaler.inverse_transform(y_test)
mae_veh  = np.mean(np.abs(y_true - y_pred))
print(f"\nPrototype (All sites) test MAE: {mae_veh:.1f} vehicles/hour")

# 9) Save
model.save(os.path.join(MODELS_DIR, MODEL_FN))
with open(os.path.join(MODELS_DIR, SCALER_FN), 'wb') as f:
    pickle.dump(scaler, f)
print(f"\n Saved prototype model → {MODELS_DIR}/{MODEL_FN}")
print(f" Saved prototype scaler → {MODELS_DIR}/{SCALER_FN}")


Using all 40 SCATS sites: [ 970 2000 2200 2820 2825 2827 2846 3001 3002 3120 3122 3126 3127 3180
 3662 3682 3685 3804 3812 4030 4032 4034 4035 4040 4043 4051 4057 4063
 4262 4263 4264 4266 4270 4272 4273 4321 4324 4335 4812 4821]
Built 401472 windows from 4 sites.
Train size: 321177 Test size: 80295


C:\Users\Admin\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                        │ (None, 24, 64)              │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 24, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_5 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
9034/9034 - 340s - 38ms/step - loss: 0.0034 - mae: 0.0401 - val_loss: 0.0017 - val_mae: 0.0278
Epoch 2/5
9034/9034 - 344s - 38ms/step - loss: 0.0017 - mae: 0.0286 - val_loss: 0.0016 - val_mae: 0.0258
Epoch 3/5
9034/9034 - 326s - 36ms/step - loss: 0.0016 - mae: 0.0274 - val_loss: 0.0014 - val_mae: 0.0240
Epoch 4/5
9034/9034 - 349s - 39ms/step - loss: 0.0013 - mae: 0.0254 - val_loss: 0.0013 - val_mae: 0.0228
Epoch 5/5
9034/9034 - 346s - 38ms/step - loss: 0.0013 - mae: 0.0248 - val_loss: 0.0013 - val_mae: 0.0233



Prototype (4 sites) test MAE: 18.7 vehicles/hour

 Saved prototype model → models/global_lstm_4sites.h5
 Saved prototype scaler → models/global_scaler_4sites.pkl
